# Day 09. Exercise 00
# Regularization

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/dayofweek.csv')

In [3]:
X = df.drop('dayofweek', axis=1)
y = df['dayofweek'] 

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=21, 
    stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

Train shape: (1348, 43)
Test shape: (338, 43)


## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

1. Подготовьте базовую модель с единственными параметрами `random_state=21`, `fit_intercept=False`.
2. Используйте многослойную перекрестную проверку в K раз с разделением на 10 частей, чтобы оценить точность модели


Результат выполнения кода, в котором вы обучили и оценили базовую модель, должен быть точно таким (используйте `%%time`, чтобы получить информацию о том, сколько времени потребовалось для запуска ячейки).:

In [5]:
def custom_cross_val(model, X, y, n_splits=10):
    skf = StratifiedKFold(n_splits=n_splits)
    valid_accs = []
    
    X_reset = X.reset_index(drop=True)
    y_reset = y.reset_index(drop=True)

    for train_index, valid_index in skf.split(X_reset, y_reset):
        X_t, X_v = X_reset.iloc[train_index], X_reset.iloc[valid_index]
        y_t, y_v = y_reset.iloc[train_index], y_reset.iloc[valid_index]
        
        model.fit(X_t, y_t)
        
        train_pred = model.predict(X_t)
        valid_pred = model.predict(X_v)
        
        train_acc = accuracy_score(y_t, train_pred)
        valid_acc = accuracy_score(y_v, valid_pred)
        
        valid_accs.append(valid_acc)
        
        print(f"train -  {train_acc:.5f}   |   valid -  {valid_acc:.5f}")
        
    print(f"Average accuracy on crossval is {np.mean(valid_accs):.5f}")
    print(f"Std is {np.std(valid_accs):.5f}")
    return np.mean(valid_accs)

In [6]:
%%time
lr_baseline = LogisticRegression(random_state=21, fit_intercept=False, max_iter=1000)
custom_cross_val(lr_baseline, X_train, y_train)

train -  0.62819   |   valid -  0.59259
train -  0.64716   |   valid -  0.62963
train -  0.63479   |   valid -  0.57037
train -  0.65540   |   valid -  0.61481
train -  0.63314   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63591   |   valid -  0.62687
Average accuracy on crossval is 0.60239
Std is 0.02852
CPU times: user 76.9 ms, sys: 5.05 ms, total: 81.9 ms
Wall time: 86.3 ms


np.float64(0.6023880597014925)

### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

1. В ячейках ниже попробуйте разные значения штрафа: "нет", "l1", "l2" – вы также можете изменить значения параметра solver.

In [7]:
print("L1")
lr_l1 = LogisticRegression(random_state=21, fit_intercept=False, penalty='l1', solver='saga', max_iter=5000)
custom_cross_val(lr_l1, X_train, y_train)

L1
train -  0.63726   |   valid -  0.58519
train -  0.64221   |   valid -  0.61481
train -  0.62984   |   valid -  0.55556
train -  0.64386   |   valid -  0.60000
train -  0.63232   |   valid -  0.57778
train -  0.63644   |   valid -  0.57778
train -  0.63644   |   valid -  0.65926
train -  0.65622   |   valid -  0.57778
train -  0.64580   |   valid -  0.58955
train -  0.63839   |   valid -  0.62687
Average accuracy on crossval is 0.59646
Std is 0.02848


np.float64(0.5964566058595908)

In [8]:
print("L2")
lr_l1 = LogisticRegression(random_state=21, fit_intercept=False, penalty='l2', solver='saga', max_iter=5000)
custom_cross_val(lr_l1, X_train, y_train)

L2
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943


np.float64(0.6016473189607519)

## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

1. Подготовьте базовую модель с единственными параметрами "probability=True", "kernel="linear", "random_state=21".
2. Используйте стратифицированную перекрестную проверку в K раз с разбиением на "10" для оценки точности модели.
3. Формат результата работы кода, в котором вы обучали и оценивали базовую модель, должен быть аналогичен тому, что вы получили для логрега.

In [9]:
%%time
svm_baseline = SVC(probability=True, kernel='linear', random_state=21)
custom_cross_val(svm_baseline, X_train, y_train)

train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556
train -  0.69415   |   valid -  0.62222
train -  0.70239   |   valid -  0.65185
train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444
train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222
train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: user 1.72 s, sys: 12.3 ms, total: 1.74 s
Wall time: 1.77 s


np.float64(0.6587064676616916)

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

1. В приведенных ниже ячейках попробуйте использовать другие значения параметра `C`.

In [10]:
C_values = [0.1, 1, 10]
for c in C_values:
    print(f"\n--- SVM with C={c} ---")
    svm_c = SVC(probability=True, kernel='linear', random_state=21, C=c)
    custom_cross_val(svm_c, X_train, y_train)


--- SVM with C=0.1 ---
train -  0.58120   |   valid -  0.55556
train -  0.57543   |   valid -  0.56296
train -  0.57378   |   valid -  0.57037
train -  0.59275   |   valid -  0.57037
train -  0.58120   |   valid -  0.54815
train -  0.57955   |   valid -  0.54815
train -  0.57296   |   valid -  0.61481
train -  0.59192   |   valid -  0.54815
train -  0.59967   |   valid -  0.52985
train -  0.57825   |   valid -  0.57463
Average accuracy on crossval is 0.56230
Std is 0.02177

--- SVM with C=1 ---
train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556
train -  0.69415   |   valid -  0.62222
train -  0.70239   |   valid -  0.65185
train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444
train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222
train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359

--- SVM with C=10 ---
train -  0.75021   | 

## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

1. Подготовьте базовую модель с единственным параметром `max_depth=10` и `random_state=21`.
2. Используйте K-кратную перекрестную проверку с разделением на 10 частей, чтобы оценить точность модели.
3. Формат результата работы кода, в котором вы обучали и оценивали базовую модель, должен быть аналогичен тому, что вы получили для логрега.

In [11]:
%%time
tree_baseline = DecisionTreeClassifier(max_depth=10, random_state=21)
custom_cross_val(tree_baseline, X_train, y_train)

train -  0.81039   |   valid -  0.74815
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.77037
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.74627
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72625
Std is 0.03635
CPU times: user 33 ms, sys: 1.28 ms, total: 34.3 ms
Wall time: 35.2 ms


np.float64(0.7262465450525151)

### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

1. В ячейках ниже попробуйте разные значения параметра "max_depth`.
2. В качестве бонуса поиграйте с другими параметрами регуляризации, пытаясь найти наилучшую комбинацию.

In [12]:
depth = 5
print(f"\n--- Decision Tree with max_depth={depth} ---")
tree_d = DecisionTreeClassifier(max_depth=depth, random_state=21)
custom_cross_val(tree_d, X_train, y_train)


--- Decision Tree with max_depth=5 ---
train -  0.59522   |   valid -  0.53333
train -  0.56307   |   valid -  0.53333
train -  0.60181   |   valid -  0.55556
train -  0.59604   |   valid -  0.57037
train -  0.60264   |   valid -  0.57778
train -  0.57955   |   valid -  0.53333
train -  0.58368   |   valid -  0.54815
train -  0.59275   |   valid -  0.51111
train -  0.58237   |   valid -  0.56716
train -  0.60132   |   valid -  0.50000
Average accuracy on crossval is 0.54301
Std is 0.02423


np.float64(0.5430127142067441)

In [13]:
depth = 10
print(f"\n--- Decision Tree with max_depth={depth} ---")
tree_d = DecisionTreeClassifier(max_depth=depth, random_state=21)
custom_cross_val(tree_d, X_train, y_train)


--- Decision Tree with max_depth=10 ---
train -  0.81039   |   valid -  0.74815
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.77037
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.74627
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72625
Std is 0.03635


np.float64(0.7262465450525151)

In [14]:
depth = 20
print(f"\n--- Decision Tree with max_depth={depth} ---")
tree_d = DecisionTreeClassifier(max_depth=depth, random_state=21)
custom_cross_val(tree_d, X_train, y_train)


--- Decision Tree with max_depth=20 ---
train -  0.98928   |   valid -  0.86667
train -  0.99011   |   valid -  0.89630
train -  0.98681   |   valid -  0.85185
train -  0.98763   |   valid -  0.90370
train -  0.98928   |   valid -  0.88148
train -  0.98186   |   valid -  0.86667
train -  0.98846   |   valid -  0.91852
train -  0.99093   |   valid -  0.89630
train -  0.99094   |   valid -  0.88060
train -  0.98847   |   valid -  0.88060
Average accuracy on crossval is 0.88427
Std is 0.01883


np.float64(0.8842675511332228)

## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

1. Обучите базовую модель с единственными параметрами `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Используйте K-кратную перекрестную проверку с разделением на 10 частей, чтобы оценить точность модели.
3. Формат результата кода, в котором вы обучали и оценивали базовую модель, должен быть похож на тот, который вы получили для логрега.

In [15]:
%%time 
rf_baseline = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
custom_cross_val(rf_baseline, X_train, y_train)

train -  0.96373   |   valid -  0.87407
train -  0.97032   |   valid -  0.91111
train -  0.96867   |   valid -  0.88889
train -  0.97279   |   valid -  0.91111
train -  0.96785   |   valid -  0.91111
train -  0.96620   |   valid -  0.85185
train -  0.96867   |   valid -  0.91111
train -  0.96702   |   valid -  0.85185
train -  0.97199   |   valid -  0.88060
train -  0.96458   |   valid -  0.85075
Average accuracy on crossval is 0.88425
Std is 0.02499
CPU times: user 424 ms, sys: 5.94 ms, total: 430 ms
Wall time: 432 ms


np.float64(0.8842454394693199)

### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

1. В новых ячейках попробуйте разные значения параметров `max_depth` и `n_estimators`.
2. В качестве бонуса поиграйте с другими параметрами регуляризации, пытаясь найти наилучшую комбинацию.

In [16]:
print("\n--- RF: n_estimators=100, max_depth=None ---")
rf_1 = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=21)
custom_cross_val(rf_1, X_train, y_train)


--- RF: n_estimators=100, max_depth=None ---
train -  1.00000   |   valid -  0.89630
train -  1.00000   |   valid -  0.94815
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.93333
train -  1.00000   |   valid -  0.91852
train -  1.00000   |   valid -  0.88889
train -  1.00000   |   valid -  0.91852
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.92537
train -  1.00000   |   valid -  0.88806
Average accuracy on crossval is 0.91394
Std is 0.01829


np.float64(0.9139358761746822)

In [17]:
print("\n--- RF: n_estimators=100, max_depth=None ---")
rf_1 = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=21)
custom_cross_val(rf_1, X_train, y_train)


--- RF: n_estimators=100, max_depth=None ---
train -  0.54905   |   valid -  0.53333
train -  0.53751   |   valid -  0.54815
train -  0.52432   |   valid -  0.50370
train -  0.53916   |   valid -  0.53333
train -  0.55812   |   valid -  0.51111
train -  0.52597   |   valid -  0.51852
train -  0.51855   |   valid -  0.52593
train -  0.54493   |   valid -  0.52593
train -  0.52059   |   valid -  0.51493
train -  0.54201   |   valid -  0.49254
Average accuracy on crossval is 0.52075
Std is 0.01529


np.float64(0.5207462686567165)

In [18]:
print("\n--- RF: n_estimators=100, max_depth=None ---")
rf_1 = RandomForestClassifier(n_estimators=50, max_depth=21, random_state=21)
custom_cross_val(rf_1, X_train, y_train)


--- RF: n_estimators=100, max_depth=None ---
train -  0.99670   |   valid -  0.89630
train -  0.99588   |   valid -  0.95556
train -  0.99835   |   valid -  0.88148
train -  0.99753   |   valid -  0.92593
train -  0.99588   |   valid -  0.91852
train -  0.99753   |   valid -  0.87407
train -  0.99835   |   valid -  0.91111
train -  0.99753   |   valid -  0.89630
train -  0.99588   |   valid -  0.90299
train -  0.99671   |   valid -  0.87313
Average accuracy on crossval is 0.90354
Std is 0.02423


np.float64(0.9035378662244333)

## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

1. Выберите наилучшую модель и используйте ее для составления прогнозов для тестового набора данных.
2. Рассчитайте конечную точность.
3. Проанализируйте: для какого дня недели ваша модель допускает наибольшее количество ошибок (в % от общего количества выборок этого класса в вашем тестовом наборе данных).
4. Сохраните модель.

In [19]:
best_model = RandomForestClassifier(n_estimators=50, max_depth=14, random_state=21)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

final_accuracy = accuracy_score(y_test, y_pred)
print(f"Final Accuracy on Test Set: {final_accuracy:.5f}")

Final Accuracy on Test Set: 0.89349


In [20]:
cm = confusion_matrix(y_test, y_pred)
classes = sorted(y_test.unique())
print("\nError Rate per Weekday:")
worst_class = None
max_error_rate = -1

for idx, label in enumerate(classes):
    total = np.sum(cm[idx, :])
    if total > 0:
        errors = total - cm[idx, idx]
        rate = (errors / total) * 100
        print(f"Class {label}: {rate:.2f}% error")
        if rate > max_error_rate:
            max_error_rate = rate
            worst_class = label

print(f"\nThe model makes the most errors for weekday: {worst_class}")


Error Rate per Weekday:
Class 0: 29.63% error
Class 1: 14.55% error
Class 2: 10.00% error
Class 3: 3.75% error
Class 4: 14.29% error
Class 5: 7.41% error
Class 6: 9.86% error

The model makes the most errors for weekday: 0


In [21]:
joblib.dump(best_model, '../models/best_model.joblib')

['../models/best_model.joblib']